# Text orientation — one-click inference

Ноутбук выбирает лучшего или именного чемпиона, возобновляет inference после прерывания и сохраняет submission, отчёт и contact sheets на Google Drive.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/text-orientation"
MODEL = "best"  # best | mobilenet_v3_large | efficientnet_b0
ARTIFACT_SOURCE = "drive"  # drive | upload
UPLOADED_PATH = None  # папка champion bundle для ARTIFACT_SOURCE="upload"
BATCH_SIZE = 128
NUM_WORKERS = 2
RESUME_INFERENCE = True
SMOKE_IMAGES = None  # 64 для проверки; None для полного submission
CONTACT_SHEET_COUNT = 16
RUN_TESTS = True

In [ ]:
import os, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")
repo_dir = Path("/content/text-orientation-classification")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
import torch

project_dir = Path(PROJECT_DIR)
test_zip = project_dir / "data" / "test.zip"
if not test_zip.is_file():
    raise FileNotFoundError(f"Положите test.zip в {test_zip}")
if ARTIFACT_SOURCE == "drive":
    if not (project_dir / "registry" / "leaderboard.json").is_file():
        raise FileNotFoundError(f"Нет registry/leaderboard.json в {project_dir}")
elif ARTIFACT_SOURCE == "upload":
    if not UPLOADED_PATH or not (Path(UPLOADED_PATH) / "champion.json").is_file():
        raise FileNotFoundError("UPLOADED_PATH должен указывать на папку с champion.json")
else:
    raise ValueError("ARTIFACT_SOURCE must be 'drive' or 'upload'")
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
command = [
    sys.executable, "-m", "scripts.infer",
    "--project-dir", str(project_dir),
    "--artifact-source", ARTIFACT_SOURCE,
    "--model", MODEL,
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--contact-sheet-count", str(CONTACT_SHEET_COUNT),
]
if ARTIFACT_SOURCE == "upload":
    command += ["--uploaded-path", str(UPLOADED_PATH)]
if SMOKE_IMAGES is not None:
    command += ["--limit", str(SMOKE_IMAGES)]
if not RESUME_INFERENCE:
    command.append("--no-resume")
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Inference failed with exit code {return_code}")

In [ ]:
runs_dir = project_dir / "inference" / "runs"
runs = sorted((path for path in runs_dir.iterdir() if path.is_dir()), key=lambda path: path.stat().st_mtime, reverse=True)
if not runs:
    raise RuntimeError("Inference run directory was not created")
latest = runs[0]
print("\nГотово:", latest)
for path in sorted(latest.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(latest))
if SMOKE_IMAGES is not None:
    print("Это smoke-запуск: submission.csv создаётся только для полного test.zip.")